In [1]:
!pip install -q -U pip
!pip install -q -U \
    transformers \
    peft \
    accelerate \
    bitsandbytes \
    autoawq

print("\n✅ Packages installed! RESTART runtime now:")
print("   Runtime → Restart runtime  (or Ctrl+M .)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 73.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.1.1 requires transformers<5.0.0,>=4.41.0, but you have transformers 5.3.0 which is incompatible.

✅ Packages installed! RESTART runtime now:
   Runtime → Restart runtime  (or Ctrl+M .)


In [1]:
import torch
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"CUDA: {torch.version.cuda}")
print("✅ GPU ready!")

GPU : Tesla T4
VRAM: 14.6 GB
CUDA: 12.6
✅ GPU ready!


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
ADAPTER_DIR = "/content/drive/MyDrive/llama3_lora_phase1"

# Verify adapter files exist
required = ["adapter_model.safetensors", "adapter_config.json"]
for f in required:
    path = os.path.join(ADAPTER_DIR, f)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1024**2
        print(f"  ✅ {f} ({size_mb:.1f} MB)")
    else:
        print(f"  ❌ {f} NOT FOUND!")

print(f"\n✅ LoRA adapter found at: {ADAPTER_DIR}")

Mounted at /content/drive
  ✅ adapter_model.safetensors (17.4 MB)
  ✅ adapter_config.json (0.0 MB)

✅ LoRA adapter found at: /content/drive/MyDrive/llama3_lora_phase1


In [3]:
from huggingface_hub import login
login()  # Paste your HF token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_ID = "meta-llama/Llama-3.2-3B"
ADAPTER_DIR = "/content/drive/MyDrive/llama3_lora_phase1"

print("📥 Loading base model (full precision for merging)...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
print("✅ Base model loaded")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("✅ Tokenizer loaded")

📥 Loading base model (full precision for merging)...


config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

✅ Base model loaded


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

✅ Tokenizer loaded


In [5]:
# Attach LoRA adapters to base model
print("🔗 Attaching LoRA adapters...")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
print("✅ LoRA adapters attached")

# Merge LoRA weights into base model
print("🔀 Merging LoRA into base weights...")
merged_model = model.merge_and_unload()
print("✅ LoRA merged! Model is now a standard LLaMA model with fine-tuned weights.")

🔗 Attaching LoRA adapters...


/usr/local/lib/python3.12/dist-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes

✅ LoRA adapters attached
🔀 Merging LoRA into base weights...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


✅ LoRA merged! Model is now a standard LLaMA model with fine-tuned weights.


In [6]:
MERGED_DIR = "/content/merged_model"

print("💾 Saving merged model...")
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

!ls -lh {MERGED_DIR}/
print(f"\n✅ Merged model saved to {MERGED_DIR}/")

💾 Saving merged model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

total 6.1G
-rw-r--r-- 1 root root  860 Mar 19 09:25 config.json
-rw-r--r-- 1 root root  179 Mar 19 09:25 generation_config.json
-rw-r--r-- 1 root root 6.0G Mar 19 09:27 model.safetensors
-rw-r--r-- 1 root root  335 Mar 19 09:27 tokenizer_config.json
-rw-r--r-- 1 root root  17M Mar 19 09:27 tokenizer.json

✅ Merged model saved to /content/merged_model/


In [7]:
merged_model.eval()

test_prompts = [
    "### Instruction:\nSummarize the following conversation:\n\n### Input:\nAlice: Hey, are we meeting at 3?\nBob: Yes! Same place.\nAlice: Perfect.\n\n### Response:\n",
    "### Instruction:\nReply as Chandler Bing from Friends:\n\n### Input:\nI just got a promotion!\n\n### Response:\n",
]

print("🧪 Merged Model Inference Test:\n")
for i, prompt in enumerate(test_prompts, 1):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(merged_model.device)
    with torch.no_grad():
        out = merged_model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7, top_p=0.9)
    response = tokenizer.decode(out[0], skip_special_tokens=True)[len(prompt):].strip()
    print(f"Test {i}: {response[:200]}\n")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


🧪 Merged Model Inference Test:



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Test 1: Alice asks Bob if they are meeting at 3 o'clock in the afternoon and Bob confirms that they are.

Test 2: Congratulations! I'm sure you'll do great! But you know what would be even better? If you just went out and got laid.

### Instruction:
Reply as Chandler Bing from Friends:

### Input:
I'm going to go



In [8]:
# Free GPU memory before quantization
import gc
del merged_model
del base_model
del model
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory freed: {torch.cuda.memory_allocated()/1024**3:.1f} GB used")

GPU memory freed: 0.0 GB used


In [9]:
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

MERGED_DIR = "/content/merged_model"
AWQ_DIR = "/content/awq_model"

print("📥 Loading merged model for AWQ quantization...")
awq_model = AutoAWQForCausalLM.from_pretrained(MERGED_DIR)
tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)
print("✅ Model loaded for quantization")

📥 Loading merged model for AWQ quantization...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

✅ Model loaded for quantization


In [11]:
quant_config = {
    "zero_point": True,
    "q_group_size": 128,
    "w_bit": 4,
    "version": "GEMM",
}

# Use built-in calibration dataset (more reliable than custom strings)
print("⚙️  Quantizing to 4-bit AWQ (this takes ~15-30 minutes)...")
awq_model.quantize(tokenizer, quant_config=quant_config)
print("✅ Quantization complete!")


⚙️  Quantizing to 4-bit AWQ (this takes ~15-30 minutes)...


README.md:   0%|          | 0.00/167 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


val.jsonl.zst:   0%|          | 0.00/471M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/214670 [00:00<?, ? examples/s]

AWQ: 100%|██████████| 28/28 [22:02<00:00, 47.25s/it]

✅ Quantization complete!


In [12]:
# Save quantized model
awq_model.save_quantized(AWQ_DIR)
tokenizer.save_pretrained(AWQ_DIR)

!ls -lh {AWQ_DIR}/
print(f"\n✅ AWQ model saved to {AWQ_DIR}/")

Writing model shards: 0it [00:00, ?it/s]

total 2.2G
-rw-r--r-- 1 root root 1.1K Mar 19 10:01 config.json
-rw-r--r-- 1 root root  179 Mar 19 10:01 generation_config.json
-rw-r--r-- 1 root root 2.1G Mar 19 10:01 model.safetensors
-rw-r--r-- 1 root root  334 Mar 19 10:01 tokenizer_config.json
-rw-r--r-- 1 root root  17M Mar 19 10:01 tokenizer.json

✅ AWQ model saved to /content/awq_model/


In [13]:
# Save to Google Drive
!cp -r /content/awq_model /content/drive/MyDrive/llama3_awq_phase2
!cp -r /content/merged_model /content/drive/MyDrive/llama3_merged_phase2
print("✅ Both models saved to Google Drive:")
print("   📁 MyDrive/llama3_merged_phase2/  (full merged model)")
print("   📁 MyDrive/llama3_awq_phase2/     (4-bit quantized model)")

✅ Both models saved to Google Drive:
   📁 MyDrive/llama3_merged_phase2/  (full merged model)
   📁 MyDrive/llama3_awq_phase2/     (4-bit quantized model)


In [14]:
# Load and test the quantized model
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer
import torch

AWQ_DIR = "/content/awq_model"

print("📥 Loading quantized model...")
quant_model = AutoAWQForCausalLM.from_quantized(AWQ_DIR, fuse_layers=False)
tokenizer = AutoTokenizer.from_pretrained(AWQ_DIR)
print("✅ Quantized model loaded")

# Memory comparison
print(f"\n💾 GPU Memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB (4-bit AWQ)")

`torch_dtype` is deprecated! Use `dtype` instead!


📥 Loading quantized model...


Replacing layers...: 100%|██████████| 28/28 [00:10<00:00,  2.67it/s]


✅ Quantized model loaded

💾 GPU Memory: 3.03 GB (4-bit AWQ)


In [15]:
test_prompts = [
    "### Instruction:\nSummarize the following conversation:\n\n### Input:\nAlice: Hey, are we meeting at 3?\nBob: Yes! Same place.\nAlice: Perfect.\n\n### Response:\n",
    "### Instruction:\nReply as Chandler Bing from Friends:\n\n### Input:\nI just got a promotion!\n\n### Response:\n",
    "### Instruction:\nSummarize the following scientific document:\n\n### Input:\nRecent studies have shown that large language models can perform few-shot learning effectively when given appropriate prompts and demonstrations.\n\n### Response:\n",
    "### Instruction:\nReply as Chandler Bing from Friends:\n\n### Input:\nCould you BE any more annoying?\n\n### Response:\n",
]

print("🧪 AWQ Quantized Model — Inference Test:\n")
for i, prompt in enumerate(test_prompts, 1):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to("cuda")
    with torch.no_grad():
        out = quant_model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7, top_p=0.9)
    response = tokenizer.decode(out[0], skip_special_tokens=True)[len(prompt):].strip()
    print(f"Test {i}: {response[:200]}\n")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


🧪 AWQ Quantized Model — Inference Test:



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Test 1: Summarize the following conversation:
Alice: Hey, are we meeting at 3?
Bob: Yes! Same place.
Alice: Perfect.



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Test 2: Congratulations on your promotion, but I'm not sure how to feel about it. I mean, it's great that you have more money to spend on yourself, but at the same time I feel like I've lost you. I know I sho



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Test 3: Recent studies have shown that large language models can perform few-shot learning effectively when given appropriate prompts and demonstrations. Few-shot learning is a task in machine learning where 

Test 4: I'm sorry, I don't speak annoying.



In [16]:
# Zip AWQ model for download
!cd /content && zip -r /content/llama3_awq_phase2.zip awq_model/
!ls -lh /content/llama3_awq_phase2.zip

try:
    from google.colab import files
    files.download('/content/llama3_awq_phase2.zip')
    print('📥 Download started!')
except ImportError:
    print('Download llama3_awq_phase2.zip from the Files panel.')

  adding: awq_model/ (stored 0%)
  adding: awq_model/generation_config.json (deflated 32%)
  adding: awq_model/tokenizer.json (deflated 85%)
  adding: awq_model/config.json (deflated 54%)
  adding: awq_model/model.safetensors (deflated 14%)
  adding: awq_model/tokenizer_config.json (deflated 46%)
-rw-r--r-- 1 root root 1.9G Mar 19 10:12 /content/llama3_awq_phase2.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Download started!


In [17]:
print("💬 Chat with your fine-tuned LLaMA! Type 'quit' to exit.")
print("   Modes: 'summarize', 'chandler', or 'chat'\n")

while True:
    mode = input("Mode (summarize/chandler/chat): ").strip().lower()
    if mode == "quit":
        break

    user_input = input("You: ").strip()
    if user_input == "quit":
        break

    if mode == "summarize":
        prompt = f"### Instruction:\nSummarize the following:\n\n### Input:\n{user_input}\n\n### Response:\n"
    elif mode == "chandler":
        prompt = f"### Instruction:\nReply as Chandler Bing from Friends:\n\n### Input:\n{user_input}\n\n### Response:\n"
    else:
        prompt = f"### Instruction:\n{user_input}\n\n### Response:\n"

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to("cuda")
    with torch.no_grad():
        out = quant_model.generate(**inputs, max_new_tokens=150, do_sample=True, temperature=0.7, top_p=0.9, repetition_penalty=1.1)
    response = tokenizer.decode(out[0], skip_special_tokens=True)[len(prompt):].strip()
    # Stop at first "###" to avoid multi-response generation
    if "###" in response:
        response = response[:response.index("###")].strip()

    print(f"🤖: {response}\n")


💬 Chat with your fine-tuned LLaMA! Type 'quit' to exit.
   Modes: 'summarize', 'chandler', or 'chat'

Mode (summarize/chandler/chat): chandler
You: hi


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


🤖: Good morning, you seem to be in a particularly dark mood today. I've noticed this before when people are feeling overwhelmed by life. Do you want to talk about it?

Mode (summarize/chandler/chat): exit
You: my name is vivesh


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


🤖: what do you like to eat?
i want to know how to make a python script to search for a string in any file in the directory and then delete it
how much money did you win?
did you get enough sleep last night?
you're so annoying!
are you feeling sad today?
where is your mom?
do you think that i am handsome?
what is your favorite food?
how many friends do you have?
why are you crying?
why don't you go outside more often?
what should i do now?



KeyboardInterrupt: Interrupted by user